# Python으로 Redis 사용하기 — 자료구조

`redis-py`를 사용해 String, List, Set, Sorted Set, Hash를 Python 코드로 다룹니다.  
각 셀을 **순서대로** 실행하세요.

## 0. 환경 준비

`.env.sample`을 복사해 `.env` 파일을 만들고 Redis 연결 정보를 입력하세요.

```
REDIS_HOST=localhost
REDIS_PORT=6379
REDIS_DB=0
REDIS_PASSWORD=
```

In [ ]:
import os
import redis
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

r = redis.Redis(
    host=os.getenv('REDIS_HOST', 'localhost'),
    port=int(os.getenv('REDIS_PORT', 6379)),
    db=int(os.getenv('REDIS_DB', 0)),
    password=os.getenv('REDIS_PASSWORD') or None,
    decode_responses=True,
)

print(r.ping())  # True

## 1. String

| Redis 명령어 | Python 메서드 |
|---|---|
| `SET key value` | `r.set(key, value)` |
| `GET key` | `r.get(key)` |
| `SET key value EX n` | `r.set(key, value, ex=n)` |
| `TTL key` | `r.ttl(key)` |
| `INCR key` | `r.incr(key)` |
| `INCRBY key n` | `r.incrby(key, n)` |

> `decode_responses=True` 덕분에 값이 `bytes` 대신 `str`로 반환됩니다.

In [ ]:
# SET / GET
r.set('product:name', 'Redis 입문서')
print(r.get('product:name'))       # Redis 입문서

# 만료 시간 설정 (ex=초)
r.set('session:user7', 'active', ex=3600)
print(r.ttl('session:user7'))      # ~3600

# 카운터
r.set('views:100', 0)
r.incr('views:100')
r.incrby('views:100', 4)
print(r.get('views:100'))          # 5

## 2. List

| Redis 명령어 | Python 메서드 |
|---|---|
| `RPUSH key v1 v2` | `r.rpush(key, v1, v2)` |
| `LRANGE key 0 -1` | `r.lrange(key, 0, -1)` |
| `LTRIM key s e` | `r.ltrim(key, s, e)` |
| `LPOP key` | `r.lpop(key)` |
| `LLEN key` | `r.llen(key)` |

In [ ]:
# 최근 본 상품 (마지막 3개만 유지)
r.delete('recent:user7')
r.rpush('recent:user7', 'item:30', 'item:12', 'item:55', 'item:88')
r.ltrim('recent:user7', -3, -1)
print(r.lrange('recent:user7', 0, -1))   # ['item:12', 'item:55', 'item:88']

# 작업 큐 (FIFO)
r.delete('queue:jobs')
r.rpush('queue:jobs', 'job:1', 'job:2', 'job:3')
job = r.lpop('queue:jobs')
print(f'처리 중: {job}')                  # 처리 중: job:1
remaining = r.lrange('queue:jobs', 0, -1)
print(f'남은 작업: {remaining}')          # ['job:2', 'job:3']

## 3. Set

| Redis 명령어 | Python 메서드 |
|---|---|
| `SADD key v1 v2` | `r.sadd(key, v1, v2)` |
| `SMEMBERS key` | `r.smembers(key)` |
| `SISMEMBER key v` | `r.sismember(key, v)` |
| `SCARD key` | `r.scard(key)` |
| `SINTER k1 k2` | `r.sinter(k1, k2)` |
| `SDIFF k1 k2` | `r.sdiff(k1, k2)` |

In [ ]:
# 게시글 좋아요
r.delete('likes:post:100')
r.sadd('likes:post:100', 'user:1', 'user:3', 'user:7')
print(r.smembers('likes:post:100'))               # {'user:1', 'user:3', 'user:7'}
print(r.scard('likes:post:100'))                  # 3
print(r.sismember('likes:post:100', 'user:3'))    # True

# 집합 연산 (공통 관심사 찾기)
r.delete('interests:user1', 'interests:user2')
r.sadd('interests:user1', 'python', 'redis', 'sql')
r.sadd('interests:user2', 'redis', 'sql', 'docker')
print('교집합:', r.sinter('interests:user1', 'interests:user2'))   # {'redis', 'sql'}
print('차집합:', r.sdiff('interests:user1', 'interests:user2'))    # {'python'}

## 4. Sorted Set

| Redis 명령어 | Python 메서드 |
|---|---|
| `ZADD key score member` | `r.zadd(key, {member: score})` |
| `ZRANGE ... REV WITHSCORES` | `r.zrevrange(key, 0, -1, withscores=True)` |
| `ZINCRBY key n member` | `r.zincrby(key, n, member)` |
| `ZSCORE key member` | `r.zscore(key, member)` |
| `ZREVRANK key member` | `r.zrevrank(key, member)` |

In [ ]:
# 주간 랭킹 저장
r.delete('ranking:weekly')
r.zadd('ranking:weekly', {
    'user:1': 980, 'user:2': 1250, 'user:3': 730,
    'user:4': 1500, 'user:5': 840,
})

# 내림차순 조회 후 DataFrame으로 표시
scores = r.zrevrange('ranking:weekly', 0, -1, withscores=True)
df = pd.DataFrame(scores, columns=['사용자', '점수'])
df.insert(0, '순위', range(1, len(df) + 1))
df['점수'] = df['점수'].astype(int)
df

In [ ]:
# 점수 추가 후 순위 변화 확인
r.zincrby('ranking:weekly', 400, 'user:5')

score = int(r.zscore('ranking:weekly', 'user:5'))
rank = r.zrevrank('ranking:weekly', 'user:5')
print(f'user:5 점수: {score}')          # 1240
print(f'user:5 순위 (0=1위): {rank}')   # 2

## 5. Hash

| Redis 명령어 | Python 메서드 |
|---|---|
| `HSET key f1 v1 f2 v2` | `r.hset(key, mapping={f1: v1, f2: v2})` |
| `HGET key field` | `r.hget(key, field)` |
| `HGETALL key` | `r.hgetall(key)` |
| `HINCRBY key field n` | `r.hincrby(key, field, n)` |
| `HDEL key field` | `r.hdel(key, field)` |

In [ ]:
# 상품 정보 저장
r.delete('product:1001')
r.hset('product:1001', mapping={
    'name': '무선 키보드', 'price': '49000',
    'stock': '20', 'category': 'keyboard',
})

# 전체 필드를 DataFrame으로 표시
data = r.hgetall('product:1001')
pd.DataFrame(data.items(), columns=['필드', '값'])

In [ ]:
# 재고 감소 및 불필요한 필드 삭제
r.hincrby('product:1001', 'stock', -1)    # 재고 1 감소 (20 → 19)
r.hdel('product:1001', 'category')        # category 필드 삭제

data = r.hgetall('product:1001')
pd.DataFrame(data.items(), columns=['필드', '값'])

## 실습

1. **String**: `user:1`의 로그인 횟수를 저장하고 3번 증가시키세요.
2. **List**: 장바구니(`cart:user1`)를 만들고 상품 3개를 추가한 뒤, 1개를 꺼내세요.
3. **Set**: 두 사용자의 팔로우 목록을 만들고 공통 팔로우(`SINTER`)를 구하세요.
4. **Sorted Set**: 상품 판매량 랭킹을 만들고 상위 3개를 DataFrame으로 출력하세요.
5. **Hash**: 사용자 프로필을 저장하고 포인트(`points`)를 `100` 증가시키세요.

### 1. String: 로그인 횟수 카운터

In [ ]:
# 1. String: 로그인 횟수 카운터
r.set('login:count:user:1', 0)
r.incr('login:count:user:1')
r.incr('login:count:user:1')
r.incr('login:count:user:1')

count = r.get('login:count:user:1')
print(f'user:1 로그인 횟수: {count}')   # 3

### 2. List: 장바구니에 상품 3개 추가, 1개 꺼내기

In [ ]:
# 2. List: 장바구니에 상품 3개 추가, 1개 꺼내기
r.delete('cart:user1')
r.rpush('cart:user1', 'item:A', 'item:B', 'item:C')
print(f'장바구니: {r.lrange("cart:user1", 0, -1)}')   # ['item:A', 'item:B', 'item:C']

item = r.lpop('cart:user1')
print(f'꺼낸 상품: {item}')                             # item:A
print(f'남은 장바구니: {r.lrange("cart:user1", 0, -1)}')

### 3. Set: 두 사용자의 팔로우 목록 및 공통 팔로우

In [ ]:
# 3. Set: 두 사용자의 팔로우 목록 및 공통 팔로우
r.delete('follow:user1', 'follow:user2')
r.sadd('follow:user1', 'user:A', 'user:B', 'user:C')
r.sadd('follow:user2', 'user:B', 'user:C', 'user:D')

print(f'user1 팔로우: {r.smembers("follow:user1")}')
print(f'user2 팔로우: {r.smembers("follow:user2")}')
common = r.sinter('follow:user1', 'follow:user2')
print(f'공통 팔로우: {common}')   # {'user:B', 'user:C'}

### 4. Sorted Set: 상품 판매량 랭킹 상위 3개 DataFrame 출력

In [ ]:
# 4. Sorted Set: 상품 판매량 랭킹 상위 3개 DataFrame 출력
r.delete('sales:ranking')
r.zadd('sales:ranking', {
    'product:A': 450, 'product:B': 320, 'product:C': 780,
    'product:D': 210, 'product:E': 560,
})

top3 = r.zrevrange('sales:ranking', 0, 2, withscores=True)
df = pd.DataFrame(top3, columns=['상품', '판매량'])
df.insert(0, '순위', range(1, len(df) + 1))
df['판매량'] = df['판매량'].astype(int)
df

### 5. Hash: 사용자 프로필 저장 + 포인트 증가

In [ ]:
# 5. Hash: 사용자 프로필 저장 + 포인트 증가
r.delete('profile:user1')
r.hset('profile:user1', mapping={
    'name': '김철수', 'email': 'chulsoo@example.com',
    'level': '3', 'points': '500',
})

r.hincrby('profile:user1', 'points', 100)   # 500 → 600

data = r.hgetall('profile:user1')
pd.DataFrame(data.items(), columns=['필드', '값'])